# WiFi CSI Based Gesture & User Identification System

Bu sistem:
- İnsan hareketini tanımlar
- Kullanıcı kimliğini belirler
- Bilinmeyen kullanıcıyı reddeder
- Yeni kullanıcıyı sisteme ekleyebilir

In [2]:
from tensorflow.keras.models import load_model
from sklearn.preprocessing import normalize
import numpy as np

In [3]:
embedding_model = load_model("../models/embedding_model.h5")



In [4]:
X = np.load("../data/X.npy")
y_user = np.load("../data/y_user.npy")

In [5]:
embeddings = embedding_model.predict(X,verbose=0)
embeddings = normalize(embeddings,axis=1)

In [6]:
train_users = [0,1,2,3,6,7]  # senin train users

gallery = {}

for user in train_users:
    user_mask = (y_user == user)
    user_embeddings = embeddings[user_mask]

    mean_embedding = np.mean(user_embeddings, axis=0)
    mean_embedding /= np.linalg.norm(mean_embedding)

    gallery[user] = mean_embedding

In [7]:
def identify(sample_embedding,gallery,threshold=0.75):
    sims ={user:np.dot(sample_embedding,gallery[user]) for user in gallery}
    best_user = max(sims,key=sims.get)
    
    if sims[best_user] > threshold:
        return best_user
    else:
        return "Unknown"

In [8]:
def enroll_user(new_usee_id,user_samples,model,gallery):
    emb = model.predict(user_samples,verbose=0)
    emb = normalize(emb,axis=1)
    
    mean_emb = np.mean(emb,axis=0)
    mean_emb /= np.linalg.norm(mean_emb)
    
    gallery[new_usee_id] = mean_emb
    

In [10]:
indices = np.where(y_user == 7)[0]
sample_index = indices[0]

sample = embeddings[sample_index]

print("Prediction:", identify(sample, gallery))
print("Gerçek:", y_user[sample_index])

sims = {user: np.dot(sample, gallery[user]) for user in gallery}
print(sims)

Prediction: 3
Gerçek: 7
{0: np.float32(0.9283277), 1: np.float32(0.8459499), 2: np.float32(0.9540025), 3: np.float32(0.95488894), 6: np.float32(0.9460733), 7: np.float32(0.94858944)}
